<a href="https://colab.research.google.com/github/rudalshan0412-code/Building-an-Artificial-Neural-Network-from-Scratch-using-NumPy/blob/main/%EB%AF%B8%EB%8B%88_%EB%87%8C_%ED%94%84%EB%A1%9C%EC%A0%9D%ED%8A%B8_05(CNN%EC%9C%BC%EB%A1%9C_%EA%B5%AC%ED%98%84).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Pytorch와 CNN을 사용하여 기존 코드를 구현할 예정이다

In [ ]:
# PyTorch CNN으로 MNIST 분류하기
# torch.nn은 torch 의 하위 패키지로 인공신경망 특화 패키지이다
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from tensorflow.keras.datasets import mnist
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# MNIST 데이터 불러오기
# MNIST 데이터는 손으로 쓴 0부터 9까지의 숫자 이미지 데이터이다

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

# (60000, 28, 28)
# (60000,)
# (10000, 28, 28)
# (10000,)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [ ]:
# 정규화
# 픽셀값은 0~255 사이이므로 255로 나누어 0~1 사이 값으로 바꾼다

x_train = x_train / 255.0
x_test = x_test / 255.0

In [ ]:
# CNN은 이미지를 1차원으로 펼치지 않는다.
# 대신 PyTorch의 Conv2d 입력 형식인 (데이터 개수, 채널 수, 높이, 너비)로 바꾼다.
# MNIST는 흑백 이미지이므로 채널 수는 1이다.

x_train = x_train.reshape(60000, 1, 28, 28)
x_test = x_test.reshape(10000, 1, 28, 28)

In [ ]:
# numpy 배열을 PyTorch Tensor로 변환한다.
# 이미지 데이터는 float32 형태로 바꾸고,
# 정답 label은 CrossEntropyLoss에서 사용하기 위해 long 타입으로 바꾼다.
# torch.long 은 torch.int64와 같은 타입이다(관습적으로 long 사용)
x_train = torch.tensor(x_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

x_test = torch.tensor(x_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [ ]:
# Dataset과 DataLoader 만들기
# TensorDataset은 이미지 데이터와 정답 데이터를 하나로 묶어준다.
# DataLoader는 size만큼 데이터를 끊어서 학습에 넣어준다.

size = 100

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=size, shuffle=False)

# 학습을 할 때에는 섞어주는 것이 좋다(나오는 순서를 규칙으로 판단할 수도 있기에)
# 정답을 맞출 때에는 굳이 섞어줄 필요는 없다

In [ ]:
# 관습적으로 클래스는 대문자로, 함수는 소문자로 시작한다.
#

In [ ]:
# CNN 모델 정의
# 기존 모델은 784 -> 128 -> 10 구조였다.
# CNN 모델은 이미지에서 특징을 뽑는 convolution 부분과,
# 최종 분류를 하는 fully connected 부분으로 나뉜다.

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 상위 클래스를 상속받는다

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        # 입력 채널 1개, 출력 채널 32개
        # kernel_size=3은 3x3 필터를 사용한다는 뜻
        # padding=1을 주면 28x28 크기가 유지된다

        self.relu1 = nn.ReLU()
        # 활성화 함수

        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        # 크기(해상도)를 줄여 연산량을 줄이며 강한 특징만을 남긴다
        # 28x28 -> 14x14로 줄어든다

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # 입력 채널 32개, 출력 채널 64개

        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        # 14x14 -> 7x7로 줄어든다

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        # conv2까지 지난 후 이미지 크기는 64채널, 7x7이다.
        # 따라서 64 * 7 * 7개를 128개로 연결한다.

        self.relu3 = nn.ReLU()

        self.fc2 = nn.Linear(128, 10)
        # 최종 출력은 숫자 0~9이므로 10개이다.

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        # CNN 결과를 fully connected layer에 넣기 위해 1차원으로 펼친다.
        # nn.Flatten의 역할을 대신한다.
        # 단, 처음부터 이미지를 펼치는 것이 아니라 convolution이 끝난 뒤 펼친다.

        x = self.fc1(x)
        x = self.relu3(x)

        x = self.fc2(x)
        # 마지막에는 softmax를 직접 쓰지 않는다.
        # CrossEntropyLoss 안에 softmax 역할이 포함되어 있다.

        return x

In [ ]:
# GPU가 가능하면 GPU를 사용하고, 아니면 CPU를 사용한다.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
# 모델, 손실 함수, 최적화 함수 정의

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
# 다중 분류 문제에서 사용하는 손실 함수
# 내부적으로 softmax + cross entropy를 같이 계산한다.

optimizer = optim.SGD(model.parameters(), lr=0.1)
# 최적화 알고리즘 패키지

In [ ]:
# 학습

epochs = 10

for epoch in range(epochs):

    model.train()
    # 모델을 학습 모드로 설정한다.

    total_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        # 순전파
        outputs = model(x_batch)

        # 손실 계산
        loss = criterion(outputs, y_batch)

        # 이전 gradient 초기화(Pytorch의 경우 기존의 gradient가 남아있기에 초기화해줘야 한다)
        optimizer.zero_grad()

        # 역전파
        loss.backward()

        # 가중치 업데이트
        optimizer.step()

        total_loss += loss.item()

        # 정확도 계산
        _, predicted = torch.max(outputs, 1)
        # outputs에서 가장 큰 값을 가진 위치가 예측한 숫자이다.

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    train_accuracy = correct / total

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}, Train Accuracy: {train_accuracy:.4f}")

Epoch [1/10], Loss: 223.6157, Train Accuracy: 0.8821
Epoch [2/10], Loss: 47.5428, Train Accuracy: 0.9754
Epoch [3/10], Loss: 32.8385, Train Accuracy: 0.9831
Epoch [4/10], Loss: 25.4176, Train Accuracy: 0.9868
Epoch [5/10], Loss: 20.4187, Train Accuracy: 0.9894
Epoch [6/10], Loss: 17.4452, Train Accuracy: 0.9912
Epoch [7/10], Loss: 14.9814, Train Accuracy: 0.9919
Epoch [8/10], Loss: 12.8094, Train Accuracy: 0.9935
Epoch [9/10], Loss: 10.8997, Train Accuracy: 0.9942
Epoch [10/10], Loss: 9.4188, Train Accuracy: 0.9950


In [ ]:
# 테스트 정확도 확인

model.eval()
# 모델을 평가 모드로 설정한다.
# nn.torch 내에 존재하는 매서드

correct = 0
total = 0

with torch.no_grad():
    # 평가할 때는 gradient 계산이 필요 없으므로 꺼준다.

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(x_batch)

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

test_accuracy = correct / total

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.9908
